#Analise e Comparação Estatística entre Métodos de Interpretabildaide / Métricas Cálculadas

In [1]:
# == Carrega as métricas, limpa os dados e ajusta colunas dos DFs por método/classificador ==

import os
import pandas as pd

DATA_BASE  = "FHS"
CLASSIFIER = ["RANDOM", "XGBOOST"]
METHOD     = ["LIME", "SHAP", "ANCHOR", "PFI", "SURROGATE"]

# Mapeia cada METHOD para (prefixo_do_df, nome_do_csv)
METHOD_CONFIG = {
    "LIME":                  ("df_lime_metrics",           "LIME_METRICS.csv"),
    "SHAP":                  ("df_shape_metrics",          "SHAP_METRICS.csv"),
    "ANCHOR":                ("df_anchor_metrics",         "ANCHOR_METRICS.csv"),
    "PFI":                   ("df_permutation_metrics",    "PFI_METRICS.csv"),
    "SURROGATE":             ("df_surrogate_tree_metrics", "SURROGATE_METRICS.csv"),
}


def verificar_ou_carregar(nome_df: str, nome_arquivo: str):
    """
    Verifica se o DataFrame já existe no ambiente global.
    Caso não exista, carrega o arquivo CSV correspondente.
    """
    if os.path.exists(nome_arquivo):
        print(f"📂 Carregando {nome_arquivo} para {nome_df}...")
        globals()[nome_df] = pd.read_csv(nome_arquivo)
    else:
        print(f"⚠ Arquivo {nome_arquivo} não encontrado. {nome_df} não será criado.")


# =========================
# 1) Carregamento
# =========================

for classifier in CLASSIFIER:
    print(f"Processando classifier: {classifier}")
    for method in METHOD:
        print(f"  Processando method: {method}")
        base_path = f"../{DATA_BASE}/notebooks/Methods/{DATA_BASE}_{method}_{classifier}/"
        print(f"    Base path: {base_path}")

        df_prefix, csv_name = METHOD_CONFIG[method]

        nome_df     = f"{df_prefix}_{classifier}"          # ex: df_lime_metrics_RANDOM
        nome_arquivo = os.path.join(base_path, csv_name)   # ex: .../WDBC_LIME_RANDOM/LIME_METRICS.csv

        verificar_ou_carregar(nome_df, nome_arquivo)


# =========================
# 2) Normalização de colunas
# =========================
# Regras:
# - dfs LIME:        renomear 'Unnamed: 0' -> 'item' e 'instancia' -> 'id'
# - dfs SHAP:        renomear 'instancia' -> 'id' e adicionar 'item' no começo (0..n-1)
# - dfs ANCHOR:      renomear 'instancia' -> 'id' e adicionar 'item' no começo (0..n-1)
# - dfs PERMUTATION: renomear 'Unnamed: 0' -> 'item' e 'instancia' -> 'id'
# - dfs SURROGATE:   renomear 'instancia' -> 'id' e adicionar 'item' no começo (0..n-1)

def _reordenar_item_primeiro(df: pd.DataFrame) -> pd.DataFrame:
    """Garante que 'item' seja a primeira coluna, se existir."""
    if "item" in df.columns:
        cols = ["item"] + [c for c in df.columns if c != "item"]
        df = df[cols]
    return df


for classifier in CLASSIFIER:
    # ---------- LIME ----------
    lime_name = f"df_lime_metrics_{classifier}"
    if lime_name in globals():
        df = globals()[lime_name].copy()

        # renomear Unnamed: 0 -> item
        for col in df.columns:
            if col.strip().lower().startswith("unnamed"):
                df = df.rename(columns={col: "item"})
                break

        # renomear instancia -> id
        if "instancia" in df.columns:
            df = df.rename(columns={"instancia": "id"})

        df = _reordenar_item_primeiro(df)
        globals()[lime_name] = df

    # ---------- SHAP ----------
    shap_name = f"df_shape_metrics_{classifier}"
    if shap_name in globals():
        df = globals()[shap_name].copy()

        # renomear instancia -> id
        if "instancia" in df.columns:
            df = df.rename(columns={"instancia": "id"})

        # adicionar item no começo (0..n-1) se não existir
        if "item" not in df.columns:
            df.insert(0, "item", range(len(df)))

        df = _reordenar_item_primeiro(df)
        globals()[shap_name] = df

    # ---------- ANCHOR ----------
    anchor_name = f"df_anchor_metrics_{classifier}"
    if anchor_name in globals():
        df = globals()[anchor_name].copy()

        # renomear instancia -> id
        if "instancia" in df.columns:
            df = df.rename(columns={"instancia": "id"})

        # adicionar item no começo (0..n-1) se não existir
        if "item" not in df.columns:
            df.insert(0, "item", range(len(df)))

        df = _reordenar_item_primeiro(df)
        globals()[anchor_name] = df

    # ---------- PERMUTATION ----------
    perm_name = f"df_permutation_metrics_{classifier}"
    if perm_name in globals():
        df = globals()[perm_name].copy()

        # renomear Unnamed: 0 -> item
        for col in df.columns:
            if col.strip().lower().startswith("unnamed"):
                df = df.rename(columns={col: "item"})
                break

        # renomear instancia -> id
        if "instancia" in df.columns:
            df = df.rename(columns={"instancia": "id"})

        df = _reordenar_item_primeiro(df)
        globals()[perm_name] = df

    # ---------- SURROGATE ----------
    sur_name = f"df_surrogate_tree_metrics_{classifier}"
    if sur_name in globals():
        df = globals()[sur_name].copy()

        # renomear instancia -> id
        if "instancia" in df.columns:
            df = df.rename(columns={"instancia": "id"})

        # adicionar item no começo (0..n-1) se não existir
        if "item" not in df.columns:
            df.insert(0, "item", range(len(df)))

        df = _reordenar_item_primeiro(df)
        globals()[sur_name] = df

print("✅ Ajustes de colunas concluídos.")


Processando classifier: RANDOM
  Processando method: LIME
    Base path: /5_xAI/databases/FHS/notebooks/Methods/FHS_LIME_RANDOM/
📂 Carregando /5_xAI/databases/FHS/notebooks/Methods/FHS_LIME_RANDOM/LIME_METRICS.csv para df_lime_metrics_RANDOM...
  Processando method: SHAP
    Base path: /5_xAI/databases/FHS/notebooks/Methods/FHS_SHAP_RANDOM/
📂 Carregando /5_xAI/databases/FHS/notebooks/Methods/FHS_SHAP_RANDOM/SHAP_METRICS.csv para df_shape_metrics_RANDOM...
  Processando method: ANCHOR
    Base path: /5_xAI/databases/FHS/notebooks/Methods/FHS_ANCHOR_RANDOM/
📂 Carregando /5_xAI/databases/FHS/notebooks/Methods/FHS_ANCHOR_RANDOM/ANCHOR_METRICS.csv para df_anchor_metrics_RANDOM...
  Processando method: PFI
    Base path: /5_xAI/databases/FHS/notebooks/Methods/FHS_PFI_RANDOM/
📂 Carregando /5_xAI/databases/FHS/notebooks/Methods/FHS_PFI_RANDOM/PFI_METRICS.csv para df_permutation_metrics_RANDOM...
  Processando method: SURROGATE
    Base path: /5_xAI/databases/FHS/notebooks/Methods/FHS_SURROGATE

In [2]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns  # opcional, mas deixei se quiser usar depois

# =========================
# Parâmetros de layout/estética
# =========================
A4_PORTRAIT  = (8.27, 11.69)
A4_LANDSCAPE = (11.69, 8.27)
ROWS_PER_PAGE = 4
HIST_MAX_BINS = 30
HIST_MIN_BINS = 5
GRID_ALPHA = 0.35
FONTSIZE_TITLE = 10
FONTSIZE_SMALL = 8
ABREV = 5  # abrevia rótulos (5 chars)

# *** Whitelist de métricas ***
RADAR_METRICS_WHITELIST = [
    "fidelity(%)", "infidelity(%)", "faithfulness(%)", "simplicity(%)",
    "consistency(%)", "robustez(%)", "completeness(%)", "soundness(%)",
    "directional soundness(%)", "stability(%)",
    "sufficiency-1(%)", "sufficiency-5(%)",
    "sufficiency-10(%)", "sufficiency-20(%)",
]

# ordem dos métodos
METHOD_ORDER = ["LIME", "SHAPLEY", "ANCHOR", "PERM IMP", "SURR TREE"]

# Base onde vão ficar os PDFs (ajuste DATA_BASE no ambiente antes)
BASE_DIR_PDF = f"../{DATA_BASE}/notebooks/Methods/XAI_analisys/pdf_metricas"
os.makedirs(BASE_DIR_PDF, exist_ok=True)

# =========================
# Utilitários
# =========================
def abbreviate(s: str, n: int = ABREV) -> str:
    return str(s)[:n].upper()

def nice_method_title(raw: str) -> str:
    return (raw.replace("df_", "")
               .replace("_metrics", "")
               .replace("_temp", "")
               .replace("_", " ")
               .strip()
               .upper())

def extract_mean_from_ci(val) -> float:
    """
    Converte:
      '28.6 [23.6, 33.6]' -> 28.6
      '28.6% [23.6, 33.6]' -> 28.6
      '28.6%' -> 28.6
      '28.6' -> 28.6
    """
    s = str(val)
    s = s.replace("%", "")
    if "[" in s:
        s = s.split("[", 1)[0]
    s = s.strip().split()[0]
    return pd.to_numeric(s, errors="coerce")

def metric_stats(x: np.ndarray) -> dict:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n == 0:
        return dict(mean=np.nan, median=np.nan, std=np.nan, min=np.nan, max=np.nan,
                    var=np.nan, cv=np.nan, n=0, ci95=np.nan)
    mean = float(np.mean(x))
    median = float(np.median(x))
    std = float(np.std(x, ddof=1)) if n > 1 else 0.0
    var = float(np.var(x, ddof=1)) if n > 1 else 0.0
    _min = float(np.min(x))
    _max = float(np.max(x))
    cv = float(std/mean*100) if mean != 0 else np.nan
    ci95 = float(1.96 * std / math.sqrt(n)) if n > 1 else 0.0
    return dict(mean=mean, median=median, std=std, min=_min, max=_max,
                var=var, cv=cv, n=n, ci95=ci95)

def clean_and_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Remove colunas 'unnamed', 'item', 'instancia', 'intancia', 'instância', 'index'
    - Remove linhas cujo índice tenha 'MÉDIA', 'MEDIA', 'MEDIANA'
    - Converte todas as colunas restantes com extract_mean_from_ci
    - Mantém apenas colunas numéricas com algum valor não nulo
    - Ordena as colunas pela RADAR_METRICS_WHITELIST, mantendo o resto no final
    """
    df = df.copy()

    # Remove colunas de índice/controle
    drop_cols = []
    for c in df.columns:
        cl = str(c).strip().lower()
        if (cl.startswith("unnamed")
            or cl in {"instancia", "intancia", "instância", "index", "item"}):
            drop_cols.append(c)
    df = df.drop(columns=drop_cols, errors="ignore")

    # Remove linhas de resumo (MÉDIA / MEDIANA) pelo índice
    idx = pd.Series(df.index.astype(str), index=df.index)
    mask_summary = idx.str.upper().str.contains("MÉDIA|MEDIA|MEDIANA")
    df = df.loc[~mask_summary].copy()

    # Converte todas as colunas com extract_mean_from_ci
    for c in df.columns:
        df[c] = df[c].apply(extract_mean_from_ci)

    # Mantém apenas colunas numéricas com algum valor não nulo
    num = df.select_dtypes(include=[np.number]).copy()
    num = num.loc[:, num.notna().any(axis=0)]

    # Ordena colunas conforme whitelist, mantendo restante ao final
    ordered = [m for m in RADAR_METRICS_WHITELIST if m in num.columns]
    ordered += [c for c in num.columns if c not in ordered]
    num = num[ordered]

    return num

# =========================
# Descobrir DataFrames por classificador (RANDOM / XGBOOST)
# =========================
def _available_methods_for_classifier(classifier: str) -> dict:
    """
    Retorna dict { 'LIME': df_num, 'SHAPLEY': df_num, ... } para o classificador.
    Assume nomes globais do tipo df_lime_metrics_RANDOM, df_shape_metrics_XGBOOST, etc.
    """
    env = globals()
    suf = classifier.upper()

    def _get_first_existing(prefix_list):
        for prefix in prefix_list:
            name = f"{prefix}_{suf}"
            df = env.get(name, None)
            if isinstance(df, pd.DataFrame) and not df.empty:
                return clean_and_numeric_df(df)
        return None

    mapping = {
        "LIME":      _get_first_existing(["df_lime_metrics"]),
        "SHAPLEY":   _get_first_existing(["df_shape_metrics"]),
        "ANCHOR":    _get_first_existing(["df_anchor_metrics"]),
        "PERM IMP":  _get_first_existing(["df_permutation_metrics"]),
        "SURR TREE": _get_first_existing(["df_surrogate_tree_metrics"]),
    }

    # remove os que não existem
    mapping = {k: v for k, v in mapping.items() if v is not None}
    return mapping

# =========================
# 4-em-1 por métrica — VERTICAL (Box | Hist | Violino | Cartão)
# =========================
def plot_metric_row(fig, parent_spec, series: pd.Series, metric_name: str):
    inner = parent_spec.subgridspec(nrows=1, ncols=4, wspace=0.25)

    x = series.to_numpy(dtype=float)
    x = x[~np.isnan(x)]
    stats = metric_stats(x)

    # 1) Boxplot
    ax1 = fig.add_subplot(inner[0, 0])
    if len(x) > 0:
        ax1.boxplot(x, vert=True, widths=0.6)
    ax1.set_title("Boxplot", fontsize=FONTSIZE_SMALL)
    ax1.grid(True, linestyle='--', alpha=GRID_ALPHA)
    ax1.set_xticks([])
    ax1.set_ylabel(metric_name, fontsize=FONTSIZE_SMALL, rotation=90)

    # 2) Histograma
    ax2 = fig.add_subplot(inner[0, 1])
    if len(x) > 0:
        bins = min(HIST_MAX_BINS, max(HIST_MIN_BINS, int(np.sqrt(len(x)))))
        ax2.hist(x, bins=bins)
    ax2.set_title("Histograma (distribuição)", fontsize=FONTSIZE_SMALL)
    ax2.grid(True, linestyle='--', alpha=GRID_ALPHA)

    # 3) Violino
    ax3 = fig.add_subplot(inner[0, 2])
    if len(x) > 0:
        ax3.violinplot(x, showmeans=True, showmedians=True)
    ax3.set_title("Violino", fontsize=FONTSIZE_SMALL)
    ax3.grid(True, linestyle='--', alpha=GRID_ALPHA)
    ax3.set_xticks([])

    # 4) Cartão de estatísticas
    ax4 = fig.add_subplot(inner[0, 3])
    ax4.axis("off")
    text = (
        f"n = {stats['n']}\n"
        f"Média = {stats['mean']:.4g}\n"
        f"Mediana = {stats['median']:.4g}\n"
        f"Desv.Padrão = {stats['std']:.4g}\n"
        f"Mínimo = {stats['min']:.4g}\n"
        f"Máximo = {stats['max']:.4g}\n"
        f"Variância = {stats['var']:.4g}\n"
        f"Coef.Var. (%) = {stats['cv']:.4g}\n"
        f"IC95% ± = {stats['ci95']:.4g}"
    )
    ax4.text(0, 0.95, "Estatísticas",
             fontsize=FONTSIZE_SMALL, fontweight="bold", va="top")
    ax4.text(0, 0.88, text, fontsize=FONTSIZE_SMALL, va="top")

def plot_metrics_grid_pages(pdf, df_num: pd.DataFrame, title: str):
    cols = list(df_num.columns)
    if not cols:
        fig = plt.figure(figsize=A4_PORTRAIT)
        ax = fig.add_subplot(1,1,1)
        ax.axis("off")
        ax.set_title(title, fontsize=FONTSIZE_TITLE, fontweight='bold')
        ax.text(0.5, 0.5, "Sem colunas numéricas para plotar.",
                ha="center", va="center", fontsize=12)
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        return

    for i in range(0, len(cols), ROWS_PER_PAGE):
        chunk = cols[i:i+ROWS_PER_PAGE]
        nrows = len(chunk)

        fig = plt.figure(figsize=A4_PORTRAIT)
        outer = fig.add_gridspec(nrows=nrows, ncols=1, hspace=0.5)

        for r, col in enumerate(chunk):
            series = df_num[col]
            plot_metric_row(fig, outer[r, 0], series, col)

        fig.suptitle(title, x=0.01, y=0.992, ha="left",
                     fontsize=FONTSIZE_TITLE, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.985])
        pdf.savefig(fig)
        plt.close(fig)

# =========================
# Radar de comparação entre métodos (1 gráfico com todos)
# =========================
def plot_radar_comparison_single(pdf, method_means: pd.DataFrame,
                                 title_prefix: str = "Comparação entre Métodos — Radar"):
    """
    - method_means: DataFrame com linhas = métodos, colunas = métricas (valores numéricos)
    Um único radar com todos os métodos juntos.
    """
    fig = plt.figure(figsize=A4_PORTRAIT)
    ax = fig.add_subplot(111, polar=True)

    if not isinstance(method_means, pd.DataFrame) or method_means.empty:
        ax.axis("off")
        ax.set_title(title_prefix, fontsize=FONTSIZE_TITLE, fontweight='bold')
        ax.text(0.5, 0.5, "method_means vazio.", ha="center", va="center", fontsize=12)
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        return

    # garante apenas colunas numéricas
    mm = method_means.select_dtypes(include=[np.number]).copy()
    if mm.empty:
        ax.axis("off")
        ax.set_title(title_prefix, fontsize=FONTSIZE_TITLE, fontweight='bold')
        ax.text(0.5, 0.5, "Sem colunas numéricas em method_means.",
                ha="center", va="center", fontsize=12)
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        return

    # Ordena colunas pela whitelist
    ordered_cols = [c for c in RADAR_METRICS_WHITELIST if c in mm.columns]
    ordered_cols += [c for c in mm.columns if c not in ordered_cols]
    mm = mm[ordered_cols]

    cols = list(mm.columns)
    n_axes = len(cols)
    if n_axes == 0:
        ax.axis("off")
        ax.set_title(title_prefix, fontsize=FONTSIZE_TITLE, fontweight='bold')
        ax.text(0.5, 0.5, "Sem métricas para o radar.", ha="center", va="center", fontsize=12)
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        return

    angles = np.linspace(0, 2*np.pi, n_axes, endpoint=False)
    angles_c = np.r_[angles, angles[0]]

    # Plota cada método (linha diferente)
    for method_name, row in mm.iterrows():
        vals = row.values.astype(float)
        vals = np.nan_to_num(vals, nan=0.0)
        vals_c = np.r_[vals, vals[0]]
        ax.plot(angles_c, vals_c, linewidth=1.6, marker="o", markersize=3,
                label=str(method_name))
        ax.fill(angles_c, vals_c, alpha=0.10)

    ax.set_xticks(angles)
    #ax.set_xticklabels([abbreviate(c) for c in cols], fontsize=FONTSIZE_SMALL)
    ax.set_xticklabels(cols, fontsize=7)
    ax.grid(True, linestyle='--', alpha=GRID_ALPHA)
    ax.set_title(title_prefix, fontsize=FONTSIZE_TITLE, fontweight='bold', pad=12)

    # legenda embaixo
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25),
              ncol=min(5, len(mm.index)), fontsize=FONTSIZE_SMALL)

    fig.tight_layout(rect=[0, 0.05, 1, 1])
    pdf.savefig(fig)
    plt.close(fig)

# =========================
# Pipeline por classificador
# =========================
def build_pdf_for_classifier(classifier: str):
    """
    Gera 1 PDF para o classificador:
      - Páginas com 4-em-1 por métrica para cada método
      - 1 página final com radar comparando métodos
    """
    methods_map = _available_methods_for_classifier(classifier)
    if not methods_map:
        print(f"⚠ Nenhum DF encontrado para o classificador {classifier}.")
        return

    pdf_path = os.path.join(
        BASE_DIR_PDF,
        f"dashboard_descr_box_hist_violino_radar_{classifier}.pdf"
    )

    with PdfPages(pdf_path) as pdf:
        # 1) Páginas por método
        for method in METHOD_ORDER:
            if method not in methods_map:
                continue
            df_num = methods_map[method]
            title = f"{method} — {classifier} — Por Métrica"
            plot_metrics_grid_pages(pdf, df_num, title=title)

        # 2) Radar comparando métodos (médias por métrica)
        #    — usando apenas métricas comuns entre os métodos disponíveis
        common_metrics = None
        for method, df_num in methods_map.items():
            cols = list(df_num.columns)
            common_metrics = set(cols) if common_metrics is None else (common_metrics & set(cols))
        common_metrics = sorted(list(common_metrics)) if common_metrics else []

        if common_metrics:
            ordered_common = [m for m in RADAR_METRICS_WHITELIST if m in common_metrics] or common_metrics
            method_means = []
            method_index = []
            for method, df_num in methods_map.items():
                method_means.append(df_num[ordered_common].mean(numeric_only=True).reindex(ordered_common))
                method_index.append(method)  # nomes LIME, SHAPLEY, etc.

            method_means = pd.DataFrame(method_means, index=method_index, columns=ordered_common)
            plot_radar_comparison_single(
                pdf,
                method_means,
                title_prefix=f"Comparação entre Métodos — Radar ({classifier})"
            )
        else:
            print(f"⚠ Não há métricas comuns entre métodos para o radar ({classifier}).")

    print(f"✅ PDF gerado ({classifier}): {pdf_path}")

# ======= EXECUÇÃO =======
build_pdf_for_classifier("RANDOM")
build_pdf_for_classifier("XGBOOST")
print("PDFs salvos em:", BASE_DIR_PDF)


/tmp/ipykernel_94165/31779276.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.985])
/tmp/ipykernel_94165/31779276.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.985])
/tmp/ipykernel_94165/31779276.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.985])
/tmp/ipykernel_94165/31779276.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.985])
/tmp/ipykernel_94165/31779276.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.985])
/tmp/ipykernel_94165/31779276.py:23

✅ PDF gerado (RANDOM): /5_xAI/databases/FHS/notebooks/Methods/XAI_analisys/pdf_metricas/dashboard_descr_box_hist_violino_radar_RANDOM.pdf


/tmp/ipykernel_94165/31779276.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.985])
/tmp/ipykernel_94165/31779276.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.985])
/tmp/ipykernel_94165/31779276.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.985])
/tmp/ipykernel_94165/31779276.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.985])
/tmp/ipykernel_94165/31779276.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.985])
/tmp/ipykernel_94165/31779276.py:23

✅ PDF gerado (XGBOOST): /5_xAI/databases/FHS/notebooks/Methods/XAI_analisys/pdf_metricas/dashboard_descr_box_hist_violino_radar_XGBOOST.pdf
PDFs salvos em: /5_xAI/databases/FHS/notebooks/Methods/XAI_analisys/pdf_metricas


In [3]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# =========================
# Configurações Gerais
# =========================
A4_PORTRAIT  = (8.27, 11.69)
ROWS_PER_PAGE = 4
GRID_ALPHA = 0.35
FONTSIZE_TITLE = 11
FONTSIZE_SMALL = 8

# Lista canônica das métricas (13 métricas)
RADAR_METRICS_WHITELIST = [
    "fidelity(%)","infidelity(%)","faithfulness(%)","simplicity(%)",
    "consistency(%)","robustez(%)","completeness(%)",
    "soundness(%)","directional soundness(%)","stability(%)",
    "sufficiency-1(%)","sufficiency-5(%)","sufficiency-10(%)",
    "sufficiency-20(%)"
]

# ==============================================
# Configuração dos dataframes fornecidos por você
# ==============================================
df_configs = [
    ("LIME",       "RANDOM",  "df_lime_metrics_RANDOM"),
    ("LIME",       "XGBOOST", "df_lime_metrics_XGBOOST"),
    ("SHAP",       "RANDOM",  "df_shape_metrics_RANDOM"),
    ("SHAP",       "XGBOOST", "df_shape_metrics_XGBOOST"),
    ("ANCHOR",     "RANDOM",  "df_anchor_metrics_RANDOM"),
    ("ANCHOR",     "XGBOOST", "df_anchor_metrics_XGBOOST"),
    ("PERMUTATION","RANDOM",  "df_permutation_metrics_RANDOM"),
    ("PERMUTATION","XGBOOST", "df_permutation_metrics_XGBOOST"),
    ("SURROGATE",  "RANDOM",  "df_surrogate_tree_metrics_RANDOM"),
    ("SURROGATE",  "XGBOOST", "df_surrogate_tree_metrics_XGBOOST"),
]

# =========================
# Funções auxiliares
# =========================
def extract_mean_from_ci(val):
    """Converte '28.6 [23, 33]' → 28.6"""
    s = str(val).replace("%", "")
    if "[" in s:
        s = s.split("[", 1)[0]
    s = s.strip()
    return pd.to_numeric(s, errors="coerce")


def clean_metrics_df(df: pd.DataFrame) -> pd.DataFrame:
    """Remove MEDIA/MÉDIA/MEDIANA, remove colunas inválidas e converte métricas p/ número."""
    df = df.copy()

    # Remove linhas MEDIA/MEDIANA
    idx_str = df.index.astype(str).str.upper()
    df = df.loc[~idx_str.str.contains("MEDIA|MÉDIA|MEDIANA")].copy()

    # Remove colunas não métricas
    drop_cols = []
    for c in df.columns:
        cl = c.lower()
        if ("item" in cl) or ("inst" in cl) or ("unnamed" in cl):
            drop_cols.append(c)
    df.drop(columns=drop_cols, inplace=True, errors="ignore")

    # Converte todas colunas numéricas via extract_mean_from_ci
    for c in df.columns:
        df[c] = df[c].apply(extract_mean_from_ci)

    # Mantém só métricas válidas
    cols_validas = [m for m in RADAR_METRICS_WHITELIST if m in df.columns]
    df = df[cols_validas]

    return df


def metric_stats(x):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return dict(mean=np.nan, median=np.nan, std=np.nan,
                    min=np.nan, max=np.nan, ci95=np.nan, n=0)
    return dict(
        mean=float(np.mean(x)),
        median=float(np.median(x)),
        std=float(np.std(x, ddof=1)),
        min=float(np.min(x)),
        max=float(np.max(x)),
        ci95=float(1.96*np.std(x, ddof=1)/np.sqrt(len(x))),
        n=len(x)
    )

# =========================
# Plot 4-em-1 por métrica
# =========================
def plot_metric_row(fig, row_spec, series, metric_name):

    inner = row_spec.subgridspec(1, 4, wspace=0.28)
    x = series.values.astype(float)
    x = x[~np.isnan(x)]
    stats = metric_stats(x)

    # Boxplot
    ax1 = fig.add_subplot(inner[0, 0])
    if len(x) > 0:
        ax1.boxplot(x, vert=True)
    ax1.set_title("Boxplot", fontsize=FONTSIZE_SMALL)
    ax1.grid(True, linestyle="--", alpha=GRID_ALPHA)
    ax1.set_ylabel(metric_name, fontsize=FONTSIZE_SMALL)

    # Histograma
    ax2 = fig.add_subplot(inner[0, 1])
    if len(x) > 0:
        bins = int(np.sqrt(len(x)))
        ax2.hist(x, bins=bins)
    ax2.set_title("Histograma", fontsize=FONTSIZE_SMALL)
    ax2.grid(True, linestyle="--", alpha=GRID_ALPHA)

    # Violino
    ax3 = fig.add_subplot(inner[0, 2])
    if len(x) > 0:
        ax3.violinplot(x, showmeans=True)
    ax3.set_title("Violino", fontsize=FONTSIZE_SMALL)
    ax3.grid(True, linestyle="--", alpha=GRID_ALPHA)

    # Cartão estatístico
    ax4 = fig.add_subplot(inner[0, 3])
    ax4.axis("off")
    txt = (
        f"n = {stats['n']}\n"
        f"Média = {stats['mean']:.3g}\n"
        f"Mediana = {stats['median']:.3g}\n"
        f"Min = {stats['min']:.3g}\n"
        f"Max = {stats['max']:.3g}\n"
        f"IC95 = ±{stats['ci95']:.3g}"
    )
    ax4.text(0, 0.95, txt, fontsize=FONTSIZE_SMALL, va="top")


# =========================
# Radar com nomes completos das métricas
# =========================
def plot_radar(pdf, dict_method_dfs, classifier):

    # Identificar métricas comuns existentes
    common_cols = None
    for df in dict_method_dfs.values():
        cols = list(df.columns)
        common_cols = set(cols) if common_cols is None else (common_cols & set(cols))

    if not common_cols:
        return

    cols = list(common_cols)

    # Construir dataframe de médias por método
    method_means = pd.DataFrame({
        method: dict_method_dfs[method][cols].mean()
        for method in dict_method_dfs
    }).T  # métodos nas linhas

    N = len(cols)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False)
    angles_closed = np.concatenate([angles, [angles[0]]])

    fig = plt.figure(figsize=A4_PORTRAIT)
    ax = fig.add_subplot(111, polar=True)

    for method, row in method_means.iterrows():
        vals = row.values.astype(float)
        vals = np.concatenate([vals, [vals[0]]])
        ax.plot(angles_closed, vals, marker="o", linewidth=1.8, label=method)
        ax.fill(angles_closed, vals, alpha=0.12)

    ax.set_xticks(angles)
    ax.set_xticklabels(cols, fontsize=7)  # ← NOME COMPLETO DA MÉTRICA

    ax.set_title(f"Radar — {classifier}", fontsize=FONTSIZE_TITLE, fontweight="bold")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=7)

    fig.tight_layout(rect=[0, 0.05, 1, 0.95])
    pdf.savefig(fig)
    plt.close(fig)


# =========================
# Construir o PDF por classificador
# =========================
def build_pdf_for_classifier(classifier):

    # Carregar todos DFs do classificador
    df_by_method = {}

    for method, clf, dfname in df_configs:
        if clf != classifier:
            continue
        if dfname in globals():
            df_raw = globals()[dfname]
            df_clean = clean_metrics_df(df_raw)
            if not df_clean.empty:
                df_by_method[method] = df_clean

    if not df_by_method:
        print(f"Sem dataframes para {classifier}")
        return

    # Caminho de saída
    outdir = f"pdf_metricas_{classifier}"
    os.makedirs(outdir, exist_ok=True)
    pdf_path = f"{outdir}/dashboard_{classifier}.pdf"

    with PdfPages(pdf_path) as pdf:

        # PÁGINAS DE 4 EM 1
        for method, df_num in df_by_method.items():

            cols = df_num.columns.tolist()
            for i in range(0, len(cols), ROWS_PER_PAGE):

                chunk = cols[i:i+ROWS_PER_PAGE]

                fig = plt.figure(figsize=A4_PORTRAIT)
                grid = fig.add_gridspec(len(chunk), 1, hspace=0.65)

                for r, col in enumerate(chunk):
                    plot_metric_row(fig, grid[r, 0], df_num[col], col)

                fig.suptitle(f"{method} — {classifier}",
                             fontsize=FONTSIZE_TITLE, fontweight="bold")

                fig.tight_layout(rect=[0, 0, 1, 0.98])
                pdf.savefig(fig)
                plt.close(fig)

        # PÁGINA FINAL: RADAR
        plot_radar(pdf, df_by_method, classifier)

    print(f"PDF gerado: {pdf_path}")


# =========================
# EXECUTAR
# =========================
build_pdf_for_classifier("RANDOM")
build_pdf_for_classifier("XGBOOST")


/tmp/ipykernel_94165/3591958723.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.98])
/tmp/ipykernel_94165/3591958723.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.98])
/tmp/ipykernel_94165/3591958723.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.98])
/tmp/ipykernel_94165/3591958723.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.98])
/tmp/ipykernel_94165/3591958723.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.98])
/tmp/ipykernel_94165/359195872

PDF gerado: pdf_metricas_RANDOM/dashboard_RANDOM.pdf


/tmp/ipykernel_94165/3591958723.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.98])
/tmp/ipykernel_94165/3591958723.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.98])
/tmp/ipykernel_94165/3591958723.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.98])
/tmp/ipykernel_94165/3591958723.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.98])
/tmp/ipykernel_94165/3591958723.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.98])
/tmp/ipykernel_94165/359195872

PDF gerado: pdf_metricas_XGBOOST/dashboard_XGBOOST.pdf


In [4]:
# ===ANOVA E TURKEY CONSIDERANDO DADO PARAMETRICO ============
# DASHBOARD ESTATÍSTICO – ANOVA + TUKEY + TAMANHO DE EFEITO
# GERA 1 PDF PARA RANDOM E 1 PDF PARA XGBOOST
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
from scipy.stats import t
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# =====================
# CONFIGURAÇÕES GERAIS
# =====================

A4_LANDSCAPE  = (11.69, 8.27)
PAGE_SIZE     = A4_LANDSCAPE
ROWS_PER_PAGE = 3
FIRST_N       = 10

BASE_PATH = f"../{DATA_BASE}/notebooks/Methods/XAI_analisys/pdf_metricas"
os.makedirs(BASE_PATH, exist_ok=True)

def output_path_for(classifier):
    return os.path.join(
        BASE_PATH,
        f"dashboard_metricas_{classifier}_anova_tukey_landscape.pdf"
    )

METHOD_COLORS = {
    "LIME": "#1f77b4","SHAPLEY":"#ff7f0e","ANCHOR":"#2ca02c",
    "PROTODASH":"#d62728","PERM IMP":"#9467bd","SURR TREE":"#8c564b",
}
METHOD_LABELS = {k:k for k in METHOD_COLORS}

METRIC_WHITELIST = [
    "fidelity(%)","infidelity(%)","faithfulness(%)","simplicity(%)","consistency(%)",
    "robustez(%)","completeness(%)","soundness(%)","directional soundness(%)","stability(%)",
    "sufficiency-1(%)","sufficiency-5(%)","sufficiency-10(%)","sufficiency-20(%)",
]

# =====================
# FUNÇÕES AUXILIARES
# =====================

def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    kill = [c for c in df.columns
            if str(c).strip().lower().startswith("unnamed")
            or str(c).strip().lower() in {"instancia","intancia","instância"}]
    return df.drop(columns=kill, errors="ignore")

def _to_float_array(series: pd.Series, first_n=FIRST_N) -> np.ndarray:
    s = series.astype(str).str.replace("%", "", regex=False)
    x = pd.to_numeric(s, errors="coerce").dropna()
    if first_n is not None:
        x = x.iloc[:first_n]
    return x.to_numpy(dtype=float)

def _mean_ci95(x: np.ndarray):
    x = np.asarray(x, dtype=float); x = x[np.isfinite(x)]
    n = len(x)
    if n == 0: return np.nan, np.nan, 0, np.nan
    mean = float(x.mean()); sd = float(x.std(ddof=1)) if n>1 else 0.0
    if n>1 and sd>0:
        se = sd/np.sqrt(n); q = t.ppf(0.975, df=n-1); ci = float(q*se)
    else:
        ci = 0.0
    return mean, ci, n, sd

def _effect_sizes_from_anova(F, df_between, df_within):
    if F is None or not np.isfinite(F) or df_between<=0 or df_within<=0:
        return np.nan, np.nan
    eta2   = (F*df_between)/(F*df_between + df_within)
    omega2 = ((F-1)*df_between)/(F*df_between + df_within + 1)
    return float(eta2), float(omega2)

def _cohen_bucket(x: float):
    if not np.isfinite(x): return "n/a", np.nan
    if x < 0.01: return "pequeno", 0.01
    elif x < 0.06: return "médio", 0.06
    else: return "grande", 0.14

# ==============================
# NOVA FUNÇÃO – CARREGA OS DFs
# ==============================

def _available_methods(classifier):
    """
    Pega automaticamente:
    df_lime_metrics_RANDOM
    df_shape_metrics_XGBOOST
    etc.
    """
    env = globals()
    suf = classifier.upper()

    def _get(name):
        df = env.get(name, None)
        return df if isinstance(df, pd.DataFrame) and not df.empty else None

    def pick(prefix):
        return _get(f"{prefix}_{suf}")

    mapping = {
        "LIME":      pick("df_lime_metrics"),
        "SHAPLEY":   pick("df_shape_metrics"),
        "ANCHOR":    pick("df_anchor_metrics"),
        "PERM IMP":  pick("df_permutation_metrics"),
        "SURR TREE": pick("df_surrogate_tree_metrics"),
        "PROTODASH": pick("df_protodash_metrics"),  # opcional
    }

    # retorna apenas os existentes
    return {k: _drop_index_cols(v) for k, v in mapping.items() if v is not None}

def _metrics_to_process(method_dfs: dict) -> list[str]:
    all_cols = set()
    for df in method_dfs.values(): all_cols.update(df.columns)
    ordered = [m for m in METRIC_WHITELIST if m in all_cols]
    return ordered or sorted(list(all_cols))

# =====================================================
# PROCESSAMENTO ESTATÍSTICO (ANOVA, TUKEY, CI95 etc.)
# =====================================================

def compute_per_metric(method_dfs: dict, metric: str):
    values   = {m: _to_float_array(df[metric], FIRST_N)
                for m, df in method_dfs.items() if metric in df.columns}

    means_ci = {m: _mean_ci95(vals) for m, vals in values.items() if len(vals)>0}

    methods_for_tests = [m for m, x in values.items() if len(x) >= 2]
    tukey_df = None
    anova = None

    if len(methods_for_tests) >= 2:
        samples = [values[m] for m in methods_for_tests]

        has_var = any(np.var(x, ddof=1) > 0 for x in samples if len(x) > 1)
        if has_var:
            F, p = stats.f_oneway(*samples)
            k = len(samples); N = sum(len(x) for x in samples)
            anova = {"F": float(F), "p": float(p), "df_between": k-1, "df_within": N-k}

            data   = np.concatenate(samples)
            labels = np.concatenate([[m]*len(values[m]) for m in methods_for_tests])
            try:
                tukey = pairwise_tukeyhsd(endog=data, groups=labels, alpha=0.05)
                df_tuk = pd.DataFrame(tukey._results_table.data[1:], columns=tukey._results_table.data[0])
                for col in ["meandiff","p-adj","lower","upper"]:
                    df_tuk[col] = pd.to_numeric(df_tuk[col], errors="coerce")
                tukey_df = df_tuk
            except Exception:
                tukey_df = None
        else:
            anova = {"F": np.nan, "p": np.nan,
                     "df_between": len(samples)-1,
                     "df_within":  sum(len(x) for x in samples)-len(samples)}

    return {"means_ci": means_ci, "anova": anova, "tukey_df": tukey_df}

# ======================================================
# DESENHO DE UMA LINHA COMPLETA DO DASHBOARD
# ======================================================

def draw_metric_row(fig, gs_row, metric_name: str, method_dfs: dict):
    res = compute_per_metric(method_dfs, metric_name)
    means_ci, anova, tukey_df = res["means_ci"], res["anova"], res["tukey_df"]

    inner = gs_row.subgridspec(
        nrows=1, ncols=6, wspace=0.06,
        width_ratios=[1.4, 0.9, 1.3, 2.05, 0.9, 1.2]
    )

    ax1 = fig.add_subplot(inner[0, 0])  
    ax2 = fig.add_subplot(inner[0, 2])  
    ax3 = fig.add_subplot(inner[0, 3])  
    ax4 = fig.add_subplot(inner[0, 5])  

    ax1.set_title(f"Métrica: {metric_name}", loc="left", fontsize=9,
                  fontweight="bold", pad=6)

    # ---- 1) BARRAS ----
    methods = [m for m in METHOD_COLORS if m in means_ci]
    if not methods:
        ax1.axis("off")
    else:
        xs   = np.arange(len(methods))
        vals = [means_ci[m][0] for m in methods]
        errs = [means_ci[m][1] for m in methods]
        cols = [METHOD_COLORS[m] for m in methods]

        bars = ax1.bar(xs, vals, yerr=errs, capsize=3,
                       color=cols, edgecolor="white",
                       linewidth=0.6, zorder=2)

        for x_i, b, m in zip(xs, bars, methods):
            ax1.text(
                x_i, b.get_height()*1.02,
                METHOD_LABELS.get(m, m),
                rotation=90, ha="center", va="bottom",
                fontsize=7, color="black"
            )

        ax1.set_xticks([])
        ax1.set_ylabel("Média (10 primeiros)", fontsize=8)
        ax1.grid(True, axis="y", linestyle="--", alpha=0.32)

    # ---- 2) TUKEY ----
    ax2.axvline(0, color="gray", ls="--", lw=0.9)
    if tukey_df is None or tukey_df.empty:
        ax2.set_yticks([])
        ax2.set_xlabel("Δ média", fontsize=8)
        ax2.set_title("Tukey HSD (n/a)", fontsize=8, pad=4)
    else:
        pairs = tukey_df.sort_values("meandiff")
        y = np.arange(len(pairs))

        ax2.hlines(y, pairs["lower"], pairs["upper"], color="black", lw=1.2)
        sig_mask = pairs["reject"].astype(bool).to_numpy()
        ax2.hlines(y[sig_mask], pairs["lower"][sig_mask], pairs["upper"][sig_mask],
                   color="#2ca02c", lw=2.0)

        ax2.plot(pairs["meandiff"], y, "ko", ms=3.5)
        labels = (pairs["group1"] + " – " + pairs["group2"]).tolist()
        ax2.set_yticks(y)
        ax2.set_yticklabels(labels, fontsize=6)
        ax2.set_xlabel("Diferença de média", fontsize=8)

    # ---- 3) ANOVA + TABELA ----
    ax3.axis("off")
    txt_lines = []

    if anova is None:
        txt_lines.append("ANOVA: n/a (amostras insuficientes)")
        eta2 = omega2 = np.nan
    else:
        F, p = anova["F"], anova["p"]
        dfb, dfw = anova["df_between"], anova["df_within"]

        if np.isfinite(F):
            txt_lines.append(f"ANOVA one-way: F({dfb}, {dfw}) = {F:.4f}, p = {p:.6f}")
            eta2, omega2 = _effect_sizes_from_anova(F, dfb, dfw)
        else:
            txt_lines.append("ANOVA: n/a (var. zero)")
            eta2 = omega2 = np.nan

    if tukey_df is not None and not tukey_df.empty:
        show = tukey_df.copy()
        for col in ["meandiff","p-adj","lower","upper"]:
            show[col] = show[col].map(lambda v: f"{float(v):.2f}")

        cols = ["group1","group2","meandiff","p-adj","lower","upper","reject"]
        header = " ".join([c.ljust(9) for c in cols])
        lines = [header, "-"*len(header)]

        for _, r in show.iterrows():
            row = " ".join([str(r[c]).ljust(9) for c in cols])
            lines.append(row)

        txt_block = "\n".join(txt_lines) + "\n\n" + "\n".join(lines)
    else:
        txt_block = "\n".join(txt_lines) + "\n\nTukey HSD: n/a"

    ax3.text(0.0, 1.0, txt_block,
             va="top", ha="left",
             family="monospace", fontsize=6.7)

    # ---- 4) ETA² / OMEGA² ----
    ax4.set_xlim(0.0, 0.155)
    ax4.set_ylim(0, 1)
    ax4.set_yticks([])
    ax4.set_xlabel("Tamanho de efeito (Cohen)", fontsize=8)
    ax4.set_title("η² / ω²", fontsize=8, pad=4)

    for x_thr, rotulo in [(0.01,"0.01"), (0.06,"0.06"), (0.14,"0.14")]:
        ax4.axvline(x_thr, color="#bdbdbd", lw=1.2)
        ax4.text(x_thr, 0.98, rotulo, rotation=90,
                 fontsize=7, ha="center", va="top", color="#6b6b6b")

    if np.isfinite(eta2):
        _, x_eta = _cohen_bucket(eta2)
        ax4.plot(x_eta, 0.66, "o", color="#1f77b4", ms=5)
        ax4.text(x_eta, 0.66, f" η={eta2:.3f}", fontsize=7.5)

    if np.isfinite(omega2):
        _, x_om = _cohen_bucket(omega2)
        ax4.plot(x_om, 0.34, "o", color="#d62728", ms=5)
        ax4.text(x_om, 0.34, f" ω={omega2:.3f}", fontsize=7.5)


# ======================================================
# GERA UM PDF PARA CADA CLASSIFICADOR
# ======================================================

def build_dashboard_pdf(classifier):
    method_dfs = _available_methods(classifier)
    method_dfs = {k: v for k, v in method_dfs.items() if v is not None}

    if not method_dfs:
        raise RuntimeError(f"Nenhum DF encontrado para o classificador {classifier}")

    output_path = output_path_for(classifier)
    metrics = _metrics_to_process(method_dfs)

    with PdfPages(output_path) as pdf:
        for start in range(0, len(metrics), ROWS_PER_PAGE):
            subset = metrics[start:start+ROWS_PER_PAGE]

            fig = plt.figure(figsize=PAGE_SIZE)
            fig.subplots_adjust(left=0.075, right=0.9, top=0.93, bottom=0.055)
            outer = fig.add_gridspec(nrows=len(subset), ncols=1, hspace=0.44)

            for r, metric in enumerate(subset):
                draw_metric_row(fig, outer[r, 0], metric, method_dfs)

            fig.suptitle(
                f"Comparação de Métricas — ANOVA/Tukey — {classifier}",
                x=0.008, y=0.975,
                ha="left", fontsize=10.5, fontweight="bold"
            )

            pdf.savefig(fig)
            plt.close(fig)

    print(f"✅ PDF salvo em: {output_path}")


# =====================
# EXECUTAR FINAL
# =====================

build_dashboard_pdf("RANDOM")
build_dashboard_pdf("XGBOOST")



✅ PDF salvo em: /5_xAI/databases/FHS/notebooks/Methods/XAI_analisys/pdf_metricas/dashboard_metricas_RANDOM_anova_tukey_landscape.pdf
✅ PDF salvo em: /5_xAI/databases/FHS/notebooks/Methods/XAI_analisys/pdf_metricas/dashboard_metricas_XGBOOST_anova_tukey_landscape.pdf


In [5]:
# ============================================================
# DASHBOARD NÃO-PARAMÉTRICO — FRIEDMAN + NEMENYI + KENDALL W
# Layout igual ao ANOVA/Tukey antigo (4 painéis)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import friedmanchisquare, rankdata, norm

# ============================================================
# CONFIGURAÇÕES GERAIS
# ============================================================

A4_LANDSCAPE  = (11.69, 8.27)
PAGE_SIZE     = A4_LANDSCAPE
ROWS_PER_PAGE = 3

BASE_PATH = f"../{DATA_BASE}/notebooks/Methods/XAI_analisys/pdf_metricas"
os.makedirs(BASE_PATH, exist_ok=True)

def output_path_for(classifier: str) -> str:
    return os.path.join(
        BASE_PATH,
        f"dashboard_metricas_{classifier}_friedman_nemenyi.pdf"
    )

METHOD_COLORS = {
    "LIME": "#1f77b4",
    "SHAPLEY": "#ff7f0e",
    "ANCHOR": "#2ca02c",
    "PERM IMP": "#9467bd",
    "SURR TREE": "#8c564b",
}
METHOD_LABELS = {k: k for k in METHOD_COLORS.keys()}

METRIC_WHITELIST = [
    "fidelity(%)","infidelity(%)","faithfulness(%)","simplicity(%)",
    "consistency(%)","robustez(%)","completeness(%)", "soundness(%)",
    "directional soundness(%)","stability(%)",
    "sufficiency-1(%)","sufficiency-5(%)",
    "sufficiency-10(%)","sufficiency-20(%)",
]

# ============================================================
# AUXILIARES
# ============================================================

def _extract_mean_from_ci(value):
    """
    Converte:
      '28.6 [23.6, 33.6]', '28.6%', '28.6% [23.6, 33.6]', '28.6'
    em 28.6 (float).
    """
    if isinstance(value, (int, float, np.number)):
        return float(value)

    if not isinstance(value, str):
        return np.nan

    v = value.replace("%", "").strip()

    if "[" in v:
        try:
            mean_part = v.split("[")[0].strip()
            return float(mean_part)
        except Exception:
            return np.nan

    try:
        return float(v)
    except Exception:
        return np.nan


def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Remove colunas de índice e ignora linhas de MÉDIA/MEDIANA."""
    # remove colunas tipo "Unnamed: 0", "instancia", etc.
    drop_cols = [
        c for c in df.columns
        if "unnamed" in c.lower() or c.lower() in {"instancia","instância","index"}
    ]
    df = df.drop(columns=drop_cols, errors="ignore")

    # remove linhas cujo índice tenha 'MÉDIA', 'MEDIA' ou 'MEDIANA'
    idx = pd.Series(df.index.astype(str), index=df.index)
    mask_summary = idx.str.upper().str.contains("MÉDIA|MEDIA|MEDIANA")
    df = df.loc[~mask_summary].copy()

    return df


def _normalize_arrays(values_dict: dict) -> dict:
    """Garante mesmo N em todos os métodos (corta no menor N)."""
    if not values_dict:
        return {}
    min_len = min(len(v) for v in values_dict.values())
    return {k: v[:min_len] for k, v in values_dict.items()}


def _cohen_bucket_W(w: float):
    """Categorias para Kendall W (heurística tipo small/medium/large)."""
    if not np.isfinite(w):
        return "n/a", 0.0
    if w < 0.1:
        return "pequeno", 0.1
    elif w < 0.3:
        return "médio", 0.3
    else:
        return "grande", 0.5

# ============================================================
# NEMENYI POST-HOC — IMPLEMENTAÇÃO PURA
# ============================================================

def nemenyi_test(rank_matrix: np.ndarray):
    """
    rank_matrix: matriz NxK de ranks (N instâncias × K métodos).
    Retorna:
      avg_ranks: vetor de K ranks médios;
      p_matrix : matriz KxK de p-values (aprox. Nemenyi).
    """
    N, k = rank_matrix.shape
    avg_ranks = rank_matrix.mean(axis=0)

    q_matrix = np.zeros((k, k), dtype=float)

    for i in range(k):
        for j in range(k):
            if i == j:
                continue
            diff = abs(avg_ranks[i] - avg_ranks[j])
            q = diff / np.sqrt(k * (k + 1) / (6 * N))
            q_matrix[i, j] = q

    # aproximação normal: p ≈ 2 * (1 - Φ(q * sqrt(2)))
    p_matrix = 2 * (1 - norm.cdf(q_matrix * np.sqrt(2.0)))

    return avg_ranks, p_matrix

# ============================================================
# CÁLCULO POR MÉTRICA
# ============================================================

def compute_nonparametric(method_dfs: dict, metric: str):
    """
    Calcula:
      - stats_desc: mean, sd por método
      - Friedman χ², p
      - Kendall W
      - Nemenyi p-values
      - Effect-size aproximado (z/√N) por par
    """
    values = {}

    for method_name, df in method_dfs.items():
        if metric not in df.columns:
            continue

        col = df[metric]

        # já removemos MÉDIA/MEDIANA em _drop_index_cols; aqui é só conversão
        arr = col.apply(_extract_mean_from_ci).to_numpy()
        arr = arr[np.isfinite(arr)]
        if len(arr) > 0:
            values[method_name] = arr

    if len(values) < 2:
        return None  # sem dados suficientes

    values = _normalize_arrays(values)
    methods = list(values.keys())
    K = len(methods)
    N = len(values[methods[0]])

    # ---------- médias / sd ----------
    stats_desc = {
        m: (float(values[m].mean()), float(values[m].std(ddof=1)))
        for m in methods
    }

    # ---------- Friedman ----------
    fried_chi2, fried_p = friedmanchisquare(*values.values())

    # ---------- Ranks ----------
    rank_matrix = np.vstack(
        [rankdata(values[m]) for m in methods]
    ).T  # N x K

    # Kendall W
    avg_rank_per_row = rank_matrix.mean(axis=1)
    S = np.sum((avg_rank_per_row - (K + 1) / 2.0) ** 2)
    kendall_W = 12 * S / (N * K * (K * K - 1))

    # ---------- Nemenyi ----------
    avg_ranks, p_matrix = nemenyi_test(rank_matrix)

    # ---------- Effect-size z/√N ----------
    effect_size = {}
    for i, m1 in enumerate(methods):
        for j, m2 in enumerate(methods):
            if i == j:
                continue
            q = abs(avg_ranks[i] - avg_ranks[j]) / np.sqrt(K * (K + 1) / (6 * N))
            z = q * np.sqrt(2.0)
            effect_size[(m1, m2)] = z / np.sqrt(N)

    # ---------- Diferenças de média (para o "forest plot") ----------
    mean_diff_rows = []
    for i, g1 in enumerate(methods):
        for j, g2 in enumerate(methods):
            if i <= j:
                continue
            m1 = stats_desc[g1][0]
            m2 = stats_desc[g2][0]
            diff = m1 - m2  # mesma convenção g1 - g2
            p_val = p_matrix[i, j]
            mean_diff_rows.append((g1, g2, diff, p_val))

    mean_diff_df = pd.DataFrame(
        mean_diff_rows,
        columns=["group1", "group2", "meandiff", "p_nemenyi"]
    ).sort_values("meandiff")

    return {
        "stats": stats_desc,
        "friedman": {"chi2": fried_chi2, "p": fried_p},
        "kendall_W": kendall_W,
        "methods": methods,
        "avg_ranks": avg_ranks,
        "nemenyi_pairs": mean_diff_df,
        "effect_size_pairs": effect_size,
        "N": N,  # 👈 número de instâncias usadas no teste
    }

# ============================================================
# DESENHO DE UMA LINHA (4 PAINÉIS)
# ============================================================

def draw_metric_row(fig, gs_row, metric_name: str, method_dfs: dict):
    res = compute_nonparametric(method_dfs, metric_name)
    if res is None:
        # sem dados suficientes
        ax = fig.add_subplot(gs_row)
        ax.axis("off")
        ax.text(0.5, 0.5, f"Sem dados suficientes para {metric_name}",
                ha="center", va="center")
        return

    stats_desc = res["stats"]
    fried = res["friedman"]
    W = res["kendall_W"]
    methods = res["methods"]
    avg_ranks = res["avg_ranks"]
    mean_diff_df = res["nemenyi_pairs"]
    N            = res["N"]

    # subgrade: [C1][sp][C2][C3][sp][C4]
    inner = gs_row.subgridspec(
        nrows=1, ncols=6, wspace=0.09,
        width_ratios=[1.9, 0.9, 1.7, 1.35, 0.9, 1.2]
    )

    ax1 = fig.add_subplot(inner[0, 0])
    ax2 = fig.add_subplot(inner[0, 2])
    ax3 = fig.add_subplot(inner[0, 3])
    ax4 = fig.add_subplot(inner[0, 5])

    # -------------------- C1: Barras --------------------
    ax1.set_title(f"Métrica: {metric_name}", loc="left",
                  fontsize=9, fontweight="bold", pad=6)

    xs = np.arange(len(methods))
    means = [stats_desc[m][0] for m in methods]
    sds   = [stats_desc[m][1] for m in methods]
    colors = [METHOD_COLORS.get(m, "#444444") for m in methods]

    bars = ax1.bar(xs, means, yerr=sds, capsize=3,
                   color=colors, edgecolor="white", linewidth=0.6, zorder=2)

    for x_i, b, m in zip(xs, bars, methods):
        ax1.text(x_i, b.get_height()*0.02, METHOD_LABELS.get(m, m),
                 rotation=90, ha="center", va="bottom",
                 fontsize=7, color="black", clip_on=True, zorder=3)

    ax1.set_xticks([])
    ax1.set_ylabel(f"Média (todas as instâncias: {N})", fontsize=8)
    ax1.grid(True, axis="y", linestyle="--", alpha=0.32, zorder=1)
    ax1.set_xlim(-0.55, len(methods) - 0.45)
    ax1.margins(x=0.0)

    # -------------------- C2: “Tukey style” Nemenyi --------------------
    ax2.axvline(0, color="gray", ls="--", lw=0.9)
    ax2.set_title("Nemenyi — pares (Δ média)", fontsize=8, pad=4)

    if mean_diff_df.empty:
        ax2.set_yticks([])
        ax2.set_xlabel("Δ média", fontsize=8)
    else:
        pairs = mean_diff_df.reset_index(drop=True)
        y = np.arange(len(pairs))

        diffs = pairs["meandiff"].to_numpy(dtype=float)
        pvals = pairs["p_nemenyi"].to_numpy(dtype=float)

        # “intervalo” visual simples: ±10% do max |diff|
        if np.any(np.isfinite(diffs)):
            span = max(1e-9, np.nanmax(np.abs(diffs)))
        else:
            span = 1.0
        half = 0.10 * span
        lower = diffs - half
        upper = diffs + half

        sig_mask = pvals < 0.05

        # segmentos
        ax2.hlines(y, lower, upper,
                   color="black", lw=1.2, zorder=1)
        ax2.hlines(y[sig_mask], lower[sig_mask], upper[sig_mask],
                   color="#2ca02c", lw=2.0, zorder=2)

        # ponto central
        ax2.plot(diffs, y, "ko", ms=3.5, zorder=3)

        labels = (pairs["group1"] + " – " + pairs["group2"]).tolist()
        ax2.set_yticks(y)
        ax2.set_yticklabels(labels, fontsize=6)
        ax2.tick_params(axis="y", pad=1)

        ax2.set_xlabel("Diferença de média (g1 - g2) | Cor: verde = p<0.05 (Nemenyi)",
                       fontsize=8)

    # -------------------- C3: Texto Friedman + tabela Nemenyi --------------------
    ax3.axis("off")

    txt_lines = []
    txt_lines.append(
        f"Friedman: χ² = {fried['chi2']:.4f},  p = {fried['p']:.8f}"
    )
    txt_lines.append(f"Kendall W = {W:.4f}")
    txt_lines.append("")
    #txt_lines.append("Ranks médios:")
    #for m, r in zip(methods, avg_ranks):
    #    txt_lines.append(f"  {m:<10} = {r:.3f}")
    #txt_lines.append("")
    txt_lines.append("Nemenyi (pares):")
    if mean_diff_df.empty:
        txt_lines.append("  n/a")
    else:
        header = "group1   group2   meandiff   p_nem    sig"
        txt_lines.append(header)
        txt_lines.append("-" * len(header))
        for _, row in mean_diff_df.iterrows():
            g1 = str(row["group1"])
            g2 = str(row["group2"])
            md = float(row["meandiff"])
            pv = float(row["p_nemenyi"])
            sig = "True" if pv < 0.05 else "False"
            txt_lines.append(
                f"{g1:<8} {g2:<8} {md:>8.2f} {pv:>8.4f} {sig:>5}"
            )

    ax3.text(0.0, 1.0, "\n".join(txt_lines),
             va="top", ha="left", family="monospace", fontsize=6.7)

    # -------------------- C4: Kendall W (tamanho de efeito global) --------------------
    ax4.set_xlim(0.0, 1.0)
    ax4.set_ylim(0, 1)
    ax4.set_yticks([])
    ax4.set_xlabel("Tamanho de efeito (Kendall W)", fontsize=8)
    ax4.set_title("W — categorias", fontsize=8, pad=4)

    for x_thr, rotulo in [(0.1, "0.1"), (0.3, "0.3"), (0.5, "0.5")]:
        ax4.axvline(x_thr, color="#bdbdbd", lw=1.2)
        ax4.text(x_thr, 0.98, rotulo, rotation=90,
                 fontsize=7, ha="center", va="top", color="#6b6b6b")

    cat, x_cat = _cohen_bucket_W(W)
    if np.isfinite(W):
        ax4.plot(x_cat, 0.5, "o", color="#1f77b4", ms=5)
        ax4.text(x_cat, 0.5, f" W={W:.3f}\n({cat})",
                 fontsize=7.5, va="center", ha="left", color="#1f77b4")

# ============================================================
# BUSCA DOS DATAFRAMES POR CLASSIFICADOR
# ============================================================

def _available_methods(classifier: str) -> dict:
    env = globals()
    suf = classifier.upper()

    def _get(name):
        df = env.get(name, None)
        return df if isinstance(df, pd.DataFrame) and not df.empty else None

    def pick(prefix):
        return _get(f"{prefix}_{suf}")

    mapping = {
        "LIME":      pick("df_lime_metrics"),
        "SHAPLEY":   pick("df_shape_metrics"),
        "ANCHOR":    pick("df_anchor_metrics"),
        "PERM IMP":  pick("df_permutation_metrics"),
        "SURR TREE": pick("df_surrogate_tree_metrics"),
    }

    return {
        k: _drop_index_cols(v)
        for k, v in mapping.items() if v is not None
    }

def _metrics_to_process(method_dfs: dict):
    all_cols = set()
    for df in method_dfs.values():
        all_cols.update(df.columns)
    # mantém ordem da whitelist
    return [m for m in METRIC_WHITELIST if m in all_cols]

# ============================================================
# GERAÇÃO DO PDF
# ============================================================

def build_dashboard_pdf(classifier: str):
    method_dfs = _available_methods(classifier)
    if not method_dfs:
        print(f"⚠️ Nenhum DF encontrado para {classifier}")
        return

    metrics = _metrics_to_process(method_dfs)
    if not metrics:
        print(f"⚠️ Nenhuma métrica encontrada para {classifier}")
        return

    output_path = output_path_for(classifier)

    with PdfPages(output_path) as pdf:
        for start in range(0, len(metrics), ROWS_PER_PAGE):
            subset = metrics[start:start+ROWS_PER_PAGE]

            fig = plt.figure(figsize=PAGE_SIZE)
            fig.subplots_adjust(left=0.075, right=0.9,
                                top=0.93, bottom=0.055)
            outer = fig.add_gridspec(
                nrows=len(subset), ncols=1, hspace=0.36
            )

            for r, metric in enumerate(subset):
                draw_metric_row(fig, outer[r, 0], metric, method_dfs)

            fig.suptitle(
                f"Dashboard Estatístico — {classifier} : Friedman + Nemenyi + Kendall W",
                x=0.008, y=0.988, ha="left",
                fontsize=10.5, fontweight="bold"
            )
            pdf.savefig(fig)
            plt.close(fig)

    print(f"✅ PDF salvo em: {os.path.abspath(output_path)}")

# ============================================================
# EXECUTAR
# ============================================================

build_dashboard_pdf("RANDOM")
build_dashboard_pdf("XGBOOST")


✅ PDF salvo em: /5_xAI/databases/FHS/notebooks/Methods/XAI_analisys/pdf_metricas/dashboard_metricas_RANDOM_friedman_nemenyi.pdf
✅ PDF salvo em: /5_xAI/databases/FHS/notebooks/Methods/XAI_analisys/pdf_metricas/dashboard_metricas_XGBOOST_friedman_nemenyi.pdf


In [6]:
# ===ANOVA E TURKEY CONSIDERANDO DADO PARAMETRICO ============
# DASHBOARD ESTATÍSTICO PARAMÉTRICO — ANOVA + TUKEY + DUTO
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.image as mpimg
from scipy import stats
from scipy.stats import t
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# ------------------------------------------------------------
# CONFIGURAÇÕES GERAIS
# ------------------------------------------------------------

A4_LANDSCAPE  = (11.69, 8.27)
PAGE_SIZE     = A4_LANDSCAPE
ROWS_PER_PAGE = 3
FIRST_N       = None   # <- usa TODAS as instâncias disponíveis

# mesma pasta das análises anteriores
BASE_DIR = f"../{DATA_BASE}/notebooks/Methods/XAI_analisys/pdf_metricas"
os.makedirs(BASE_DIR, exist_ok=True)

def output_path_for(classifier: str) -> str:
    return os.path.join(
        BASE_DIR,
        f"dashboard_metricas_{classifier}_anova_tukey_duto.pdf"
    )

DUTO_IMG = "duto3d.png"   # png do duto 3D (mesma pasta do notebook/script)

METHOD_COLORS = {
    "LIME":      "#1f77b4",
    "SHAPLEY":   "#ff7f0e",
    "ANCHOR":    "#2ca02c",
    "PERM IMP":  "#9467bd",
    "SURR TREE": "#8c564b",
    # "PROTODASH": "#d62728",  # se quiser ativar depois, é só incluir no mapping
}

METHOD_LABELS = {k: k for k in METHOD_COLORS.keys()}

METRIC_WHITELIST = [
    "fidelity(%)", "infidelity(%)", "faithfulness(%)", "simplicity(%)",
    "consistency(%)", "robustez(%)", "completeness(%)", "soundness(%)",
    "directional soundness(%)", "stability(%)",
    "sufficiency-1(%)", "sufficiency-5(%)",
    "sufficiency-10(%)", "sufficiency-20(%)",
]

# ------------------------------------------------------------
# AUXILIARES
# ------------------------------------------------------------

def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove colunas de índice e também linhas de resumo
    (MÉDIA / MEDIANA) pela label do índice.
    """
    # colunas lixo
    kill = [
        c for c in df.columns
        if str(c).strip().lower().startswith("unnamed")
        or str(c).strip().lower() in {"instancia", "intancia", "instância", "index"}
    ]
    df = df.drop(columns=kill, errors="ignore")

    # remove linhas de resumo (MÉDIA / MEDIANA)
    idx = pd.Series(df.index.astype(str), index=df.index)
    mask_summary = idx.str.upper().str.contains("MÉDIA|MEDIA|MEDIANA")
    df = df.loc[~mask_summary].copy()

    return df

def _extract_mean_from_ci(val) -> float:
    """
    Converte strings do tipo:
      '28.6 [23.6, 33.6]' ou '28.6' ou '28.6%'
    para float (28.6).
    """
    s = str(val)
    # tira percentuais
    s = s.replace("%", "")
    # se tiver '[', pega só antes do '['
    if "[" in s:
        s = s.split("[", 1)[0]
    # se tiver espaço e quiser só o primeiro token numérico
    s = s.strip().split()[0]
    return pd.to_numeric(s, errors="coerce")

def _to_float_array(series: pd.Series, first_n=FIRST_N) -> np.ndarray:
    x = series.apply(_extract_mean_from_ci).dropna()
    if first_n is not None:
        x = x.iloc[:first_n]
    return x.to_numpy(dtype=float)

def _mean_ci95(x: np.ndarray):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n == 0:
        return np.nan, np.nan, 0, np.nan
    mean = float(x.mean())
    sd   = float(x.std(ddof=1)) if n > 1 else 0.0
    if n > 1 and sd > 0:
        se = sd / np.sqrt(n)
        q  = t.ppf(0.975, df=n - 1)
        ci = float(q * se)
    else:
        ci = 0.0
    return mean, ci, n, sd

def _effect_sizes_from_anova(F, df_between, df_within):
    if F is None or not np.isfinite(F) or df_between <= 0 or df_within <= 0:
        return np.nan, np.nan
    eta2   = (F * df_between) / (F * df_between + df_within)
    omega2 = ((F - 1) * df_between) / (F * df_between + df_within + 1)
    return float(eta2), float(omega2)

def _cohen_bucket(x: float):
    if not np.isfinite(x):
        return "n/a", np.nan
    if x < 0.01:
        return "pequeno", 0.01
    elif x < 0.06:
        return "médio", 0.06
    else:
        return "grande", 0.14

# ------------------------------------------------------------
# DISPONIBILIDADE DE MÉTODOS POR CLASSIFICADOR
# ------------------------------------------------------------

def _available_methods(classifier: str) -> dict:
    """
    Carrega automaticamente os DataFrames para um classificador:
    - classifier: 'RANDOM' ou 'XGBOOST'
    Espera encontrar:
      df_lime_metrics_<CLASS>
      df_shape_metrics_<CLASS>
      df_anchor_metrics_<CLASS>
      df_permutation_metrics_<CLASS>
      df_surrogate_tree_metrics_<CLASS>
    (e opcionalmente df_protodash_metrics_<CLASS>)
    """
    env = globals()
    suf = classifier.upper()

    def _get(name):
        df = env.get(name, None)
        if isinstance(df, pd.DataFrame) and not df.empty:
            return _drop_index_cols(df)
        return None

    def pick(prefix):
        return _get(f"{prefix}_{suf}")

    mapping = {
        "LIME":      pick("df_lime_metrics"),
        "SHAPLEY":   pick("df_shape_metrics"),   # ou df_shap_metrics_*
        "ANCHOR":    pick("df_anchor_metrics"),
        "PERM IMP":  pick("df_permutation_metrics"),
        "SURR TREE": pick("df_surrogate_tree_metrics"),
        # "PROTODASH": pick("df_protodash_metrics"),  # se existir
    }

    # também tenta variações SHAP se o principal não existir
    if mapping["SHAPLEY"] is None:
        alt = env.get(f"df_shap_metrics_{suf}", None) or env.get(f"df_shapley_metrics_{suf}", None)
        if isinstance(alt, pd.DataFrame) and not alt.empty:
            mapping["SHAPLEY"] = _drop_index_cols(alt)

    return {k: v for k, v in mapping.items() if v is not None}

def _metrics_to_process(method_dfs: dict) -> list[str]:
    all_cols = set()
    for df in method_dfs.values():
        all_cols.update(df.columns)
    ordered = [m for m in METRIC_WHITELIST if m in all_cols]
    return ordered or sorted(list(all_cols))

# ------------------------------------------------------------
# CÁLCULO PARAMÉTRICO POR MÉTRICA (ANOVA + TUKEY)
# ------------------------------------------------------------

def compute_per_metric(method_dfs: dict, metric: str):
    values = {
        m: _to_float_array(df[metric], FIRST_N)
        for m, df in method_dfs.items()
        if metric in df.columns
    }

    # descarta métodos sem dados
    values = {m: v for m, v in values.items() if len(v) > 0}

    means_ci = {
        m: _mean_ci95(vals)
        for m, vals in values.items()
        if len(vals) > 0
    }

    methods_for_tests = [m for m, x in values.items() if len(x) >= 2]

    tukey_df = None
    anova    = None

    if len(methods_for_tests) >= 2:
        samples = [values[m] for m in methods_for_tests]
        has_var = any(np.var(x, ddof=1) > 0 for x in samples if len(x) > 1)

        if has_var:
            F, p = stats.f_oneway(*samples)
            k = len(samples)
            N = sum(len(x) for x in samples)
            anova = {
                "F": float(F),
                "p": float(p),
                "df_between": k - 1,
                "df_within":  N - k,
            }

            data   = np.concatenate(samples)
            labels = np.concatenate([[m] * len(values[m]) for m in methods_for_tests])

            try:
                tukey = pairwise_tukeyhsd(endog=data, groups=labels, alpha=0.05)
                df_tuk = pd.DataFrame(
                    tukey._results_table.data[1:],
                    columns=tukey._results_table.data[0]
                )
                for col in ["meandiff", "p-adj", "lower", "upper"]:
                    df_tuk[col] = pd.to_numeric(df_tuk[col], errors="coerce")
                tukey_df = df_tuk
            except Exception:
                tukey_df = None
        else:
            anova = {
                "F": np.nan,
                "p": np.nan,
                "df_between": len(samples) - 1,
                "df_within":  sum(len(x) for x in samples) - len(samples),
            }

    return {"means_ci": means_ci, "anova": anova, "tukey_df": tukey_df}

# ------------------------------------------------------------
# COLUNA 5: DUTO PNG + RANKING DOS MÉTODOS
# ------------------------------------------------------------

def _draw_duto_ranking(ax, means_ci: dict, image_path: str = DUTO_IMG):
    """
    Coluna 5: duto 3D com PNG + rótulos dos métodos ordenados
    por média (maior -> menor).
    """
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    if not means_ci:
        ax.text(0.02, 0.5, "n/a", fontsize=8, va="center", ha="left")
        return

    medias = {m: v[0] for m, v in means_ci.items() if np.isfinite(v[0])}
    if not medias:
        ax.text(0.02, 0.5, "n/a", fontsize=8, va="center", ha="left")
        return

    ordered = sorted(medias.items(), key=lambda kv: kv[1], reverse=True)
    methods = [m.upper() for m, _ in ordered]

    if not os.path.exists(image_path):
        ax.text(0.02, 0.5, f"Imagem não encontrada:\n{image_path}",
                fontsize=8, va="center", ha="left")
        return

    img = mpimg.imread(image_path)

    # posição do duto na área 0..1
    left, right = 0.04, 0.34
    bottom, top = 0.05, 0.95
    ax.imshow(img, extent=[left, right, bottom, top],
              aspect="auto", zorder=-1)

    # + e - ao lado
    sym_x = right + 0.05
    ax.text(sym_x, top - 0.01, "+", fontsize=10, fontweight="bold",
            ha="center", va="top", color="black")
    ax.text(sym_x, bottom + 0.01, "−", fontsize=10, fontweight="bold",
            ha="center", va="bottom", color="black")

    # labels ordenados verticalmente
    label_x = right + 0.14
    y_top = top - 0.08
    y_bot = bottom + 0.08
    ys = np.linspace(y_top, y_bot, len(methods))
    for y, m in zip(ys, methods):
        ax.text(label_x, y, m, fontsize=7, fontweight="bold",
                ha="left", va="center", color="black")

# ------------------------------------------------------------
# DESENHO 1 LINHA (5 QUADRANTES + 3 ESPAÇADORES)
# ------------------------------------------------------------

def draw_metric_row(fig, gs_row, metric_name: str, method_dfs: dict):
    res = compute_per_metric(method_dfs, metric_name)
    means_ci, anova, tukey_df = res["means_ci"], res["anova"], res["tukey_df"]

    # 8 colunas: [C1][SP12][C2][C3][SP34][C4][SP45][C5]
    inner = gs_row.subgridspec(
        nrows=1, ncols=8, wspace=0.05,
        width_ratios=[1.30, 0.94, 1.10, 2.05, 1.20, 1.00, 0.30, 1.25]
    )

    ax1 = fig.add_subplot(inner[0, 0])  # Barras
    ax2 = fig.add_subplot(inner[0, 2])  # Tukey
    ax3 = fig.add_subplot(inner[0, 3])  # ANOVA + tabela
    ax4 = fig.add_subplot(inner[0, 5])  # η² / ω²
    ax5 = fig.add_subplot(inner[0, 7])  # Duto

    # título da linha
    ax1.set_title(f"Métrica: {metric_name}",
                  loc="left", fontsize=9, fontweight="bold", pad=6)

    # ---------- (1) Barras ----------
    methods = [m for m in METHOD_COLORS if m in means_ci]
    if not methods:
        ax1.axis("off")
        ax1.text(0.02, 0.5, "Sem dados.", ha="left", va="center")
    else:
        xs   = np.arange(len(methods))
        vals = [means_ci[m][0] for m in methods]
        errs = [means_ci[m][1] for m in methods]
        cols = [METHOD_COLORS[m] for m in methods]

        bars = ax1.bar(xs, vals, yerr=errs, capsize=3,
                       color=cols, edgecolor="white",
                       linewidth=0.6, zorder=2)

        for x_i, b, m in zip(xs, bars, methods):
            ax1.text(x_i, b.get_height() * 0.02,
                     METHOD_LABELS.get(m, m),
                     rotation=90, ha="center", va="bottom",
                     fontsize=7, color="black",
                     clip_on=True, zorder=3)

        ax1.set_xticks([])
        ax1.set_ylabel("Média (todas as instâncias)", fontsize=8)
        ax1.grid(True, axis="y", linestyle="--", alpha=0.32, zorder=1)
        ax1.set_xlim(-0.55, len(methods) - 0.45)
        ax1.margins(x=0.0)

    # ---------- (2) Tukey ----------
    ax2.axvline(0, color="gray", ls="--", lw=0.9)
    if tukey_df is None or tukey_df.empty:
        ax2.set_yticks([])
        ax2.set_xlabel("Δ média", fontsize=8)
        ax2.set_title("Tukey HSD (n/a)", fontsize=8, pad=4)
    else:
        pairs = tukey_df.sort_values("meandiff")
        y = np.arange(len(pairs))

        ax2.hlines(y, pairs["lower"], pairs["upper"],
                   color="black", lw=1.2, zorder=1)
        sig_mask = pairs["reject"].astype(bool).to_numpy()
        ax2.hlines(y[sig_mask], pairs["lower"][sig_mask],
                   pairs["upper"][sig_mask],
                   color="#2ca02c", lw=2.0, zorder=2)
        ax2.plot(pairs["meandiff"], y, "ko", ms=3.5, zorder=3)

        labels = (pairs["group1"] + " – " + pairs["group2"]).tolist()
        ax2.set_yticks(y)
        ax2.set_yticklabels(labels, fontsize=6)
        ax2.tick_params(axis="y", pad=1)
        ax2.set_xlabel("Diferença de média (g1 - g2)", fontsize=8)

    # ---------- (3) ANOVA + tabela ----------
    ax3.axis("off")
    txt_lines = []
    if anova is None:
        txt_lines.append("ANOVA: n/a (amostras insuficientes).")
        eta2 = omega2 = np.nan
    else:
        F   = anova["F"]
        p   = anova["p"]
        dfb = anova["df_between"]
        dfw = anova["df_within"]

        if np.isfinite(F):
            txt_lines.append(
                f"ANOVA one-way: F({dfb}, {dfw}) = {F:.4f},  p = {p:.8f}"
            )
            eta2, omega2 = _effect_sizes_from_anova(F, dfb, dfw)
        else:
            txt_lines.append("ANOVA: n/a (variância zero).")
            eta2 = omega2 = np.nan

    if tukey_df is not None and not tukey_df.empty:
        show = tukey_df.copy()
        show["meandiff"] = show["meandiff"].map(lambda v: f"{float(v):.2f}")
        show["p-adj"]    = show["p-adj"].map(lambda v: f"{float(v):.4f}")
        show["lower"]    = show["lower"].map(lambda v: f"{float(v):.2f}")
        show["upper"]    = show["upper"].map(lambda v: f"{float(v):.2f}")
        show["reject"]   = show["reject"].map(lambda v: "True" if bool(v) else "False")

        cols = ["group1", "group2", "meandiff", "p-adj", "lower", "upper", "reject"]
        header = " ".join([c.ljust(9) for c in cols])
        lines  = [header, "-" * len(header)]

        for _, r in show.iterrows():
            row = " ".join([str(r[c]).ljust(9) for c in cols])
            lines.append(row)

        txt_block = "\n".join(txt_lines) + "\n\n" + "\n".join(lines)
    else:
        txt_block = "\n".join(txt_lines) + "\n\nTukey HSD: n/a"

    ax3.text(0.0, 1.0, txt_block,
             va="top", ha="left",
             family="monospace", fontsize=6.7)

    # ---------- (4) η² / ω² ----------
    ax4.set_xlim(0.0, 0.155)
    ax4.set_ylim(0, 1)
    ax4.set_yticks([])
    ax4.set_xlabel("Tamanho de efeito (Cohen)", fontsize=8)
    ax4.set_title("η² / ω² — categorias", fontsize=8, pad=4)

    for x_thr, rotulo in [(0.01, "0.01"), (0.06, "0.06"), (0.14, "0.14")]:
        ax4.axvline(x_thr, color="#bdbdbd", lw=1.2)
        ax4.text(x_thr, 0.98, rotulo, rotation=90,
                 fontsize=7, ha="center", va="top",
                 color="#6b6b6b")

    if anova is not None and np.isfinite(anova.get("F", np.nan)):
        eta2, omega2 = _effect_sizes_from_anova(
            anova["F"], anova["df_between"], anova["df_within"]
        )
    else:
        eta2 = omega2 = np.nan

    if np.isfinite(eta2):
        _, x_eta = _cohen_bucket(eta2)
        ax4.plot(x_eta, 0.66, "o", color="#1f77b4", ms=5)
        ax4.text(x_eta, 0.66, f" η={eta2:.3f}",
                 fontsize=7.5, va="center", ha="left", color="#1f77b4")

    if np.isfinite(omega2):
        _, x_om = _cohen_bucket(omega2)
        ax4.plot(x_om, 0.34, "o", color="#d62728", ms=5)
        ax4.text(x_om, 0.34, f" ω={omega2:.3f}",
                 fontsize=7.5, va="center", ha="left", color="#d62728")

    # ---------- (5) Duto + ranking ----------
    _draw_duto_ranking(ax5, means_ci, image_path=DUTO_IMG)

# ------------------------------------------------------------
# GERAÇÃO DO PDF POR CLASSIFICADOR
# ------------------------------------------------------------

def build_dashboard_pdf(classifier: str):
    method_dfs = _available_methods(classifier)
    if not method_dfs:
        print(f"⚠️ Nenhum DataFrame de método encontrado para {classifier}.")
        return

    metrics = _metrics_to_process(method_dfs)
    if not metrics:
        print(f"⚠️ Nenhuma métrica reconhecida encontrada para {classifier}.")
        return

    output_path = output_path_for(classifier)
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)

    with PdfPages(output_path) as pdf:
        for start in range(0, len(metrics), ROWS_PER_PAGE):
            subset = metrics[start:start + ROWS_PER_PAGE]

            fig = plt.figure(figsize=PAGE_SIZE)
            fig.subplots_adjust(left=0.075, right=0.975,
                                top=0.90, bottom=0.06)

            outer = fig.add_gridspec(
                nrows=len(subset), ncols=1, hspace=0.44
            )

            for r, metric in enumerate(subset):
                draw_metric_row(fig, outer[r, 0], metric, method_dfs)

            fig.suptitle(
                f"Comparação de Métricas — ANOVA/Tukey, η²/ω² e Ranking (duto)\nClassificador: {classifier}",
                x=0.008, y=0.965, ha="left",
                fontsize=10.5, fontweight="bold"
            )

            pdf.savefig(fig)
            plt.close(fig)

    print(f"✅ PDF salvo em: {output_path}")

# ------------------------------------------------------------
# EXECUTAR PARA RANDOM E XGBOOST
# ------------------------------------------------------------

build_dashboard_pdf("RANDOM")
build_dashboard_pdf("XGBOOST")


✅ PDF salvo em: /5_xAI/databases/FHS/notebooks/Methods/XAI_analisys/pdf_metricas/dashboard_metricas_RANDOM_anova_tukey_duto.pdf
✅ PDF salvo em: /5_xAI/databases/FHS/notebooks/Methods/XAI_analisys/pdf_metricas/dashboard_metricas_XGBOOST_anova_tukey_duto.pdf


In [7]:
# ============================================================
# DASHBOARD NÃO-PARAMÉTRICO — FRIEDMAN + NEMENYI + DUTO
# Layout igual ao ANOVA/Tukey antigo (5 painéis)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.image as mpimg
from scipy.stats import friedmanchisquare, rankdata, norm

# ------------------------------------------------------------
# CONFIGURAÇÕES GERAIS
# ------------------------------------------------------------

A4_LANDSCAPE  = (11.69, 8.27)
PAGE_SIZE     = A4_LANDSCAPE
ROWS_PER_PAGE = 3
FIRST_N       = None   # usa TODAS as instâncias válidas

BASE_DIR = f"../{DATA_BASE}/notebooks/Methods/XAI_analisys/pdf_metricas"
os.makedirs(BASE_DIR, exist_ok=True)

def output_path_for(classifier: str) -> str:
    return os.path.join(
        BASE_DIR,
        f"dashboard_metricas_{classifier}_friedman_nemenyi_duto.pdf"
    )

DUTO_IMG = "duto3d.png"   # png do duto 3D (mesma pasta do notebook/script)

METHOD_COLORS = {
    "LIME":      "#1f77b4",
    "SHAPLEY":   "#ff7f0e",
    "ANCHOR":    "#2ca02c",
    "PERM IMP":  "#9467bd",
    "SURR TREE": "#8c564b",
    # "PROTODASH": "#d62728",  # se quiser ativar depois
}

METHOD_LABELS = {k: k for k in METHOD_COLORS.keys()}

METRIC_WHITELIST = [
    "fidelity(%)", "infidelity(%)", "faithfulness(%)", "simplicity(%)",
    "consistency(%)", "robustez(%)", "completeness(%)", "soundness(%)",
    "directional soundness(%)", "stability(%)",
    "sufficiency-1(%)", "sufficiency-5(%)",
    "sufficiency-10(%)", "sufficiency-20(%)",
]

# ------------------------------------------------------------
# AUXILIARES
# ------------------------------------------------------------

def _extract_mean_from_ci(value):
    """
    Converte:
      '28.6 [23.6, 33.6]', '28.6%', '28.6% [23.6, 33.6]', '28.6'
    em 28.6 (float).
    """
    if isinstance(value, (int, float, np.number)):
        return float(value)

    if not isinstance(value, str):
        return np.nan

    s = value.replace("%", "").strip()

    if "[" in s:
        try:
            mean_part = s.split("[", 1)[0].strip()
            return float(mean_part)
        except Exception:
            return np.nan

    try:
        return float(s.split()[0])
    except Exception:
        return np.nan


def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove colunas de índice e também linhas de resumo
    (MÉDIA / MEDIANA) pela label do índice.
    """
    # colunas lixo
    kill = [
        c for c in df.columns
        if str(c).strip().lower().startswith("unnamed")
        or str(c).strip().lower() in {"instancia", "intancia", "instância", "index"}
    ]
    df = df.drop(columns=kill, errors="ignore")

    # remove linhas de resumo (MÉDIA / MEDIA / MEDIANA)
    idx = pd.Series(df.index.astype(str), index=df.index)
    mask_summary = idx.str.upper().str.contains("MÉDIA|MEDIA|MEDIANA")
    df = df.loc[~mask_summary].copy()

    return df


def _normalize_arrays(values_dict: dict) -> dict:
    """Garante mesmo N em todos os métodos (corta no menor N)."""
    if not values_dict:
        return {}
    min_len = min(len(v) for v in values_dict.values())
    return {k: v[:min_len] for k, v in values_dict.items()}


def _cohen_bucket_W(w: float):
    """Categorias para Kendall W (heurística small/medium/large)."""
    if not np.isfinite(w):
        return "n/a", 0.0
    if w < 0.1:
        return "pequeno", 0.1
    elif w < 0.3:
        return "médio", 0.3
    else:
        return "grande", 0.5

# ------------------------------------------------------------
# NEMENYI POST-HOC — IMPLEMENTAÇÃO PURA
# ------------------------------------------------------------

def nemenyi_test(rank_matrix: np.ndarray):
    """
    rank_matrix: matriz NxK de ranks (N instâncias × K métodos).
    Retorna:
      avg_ranks: vetor de K ranks médios;
      p_matrix : matriz KxK de p-values (aprox. Nemenyi).
    """
    N, k = rank_matrix.shape
    avg_ranks = rank_matrix.mean(axis=0)

    q_matrix = np.zeros((k, k), dtype=float)

    for i in range(k):
        for j in range(k):
            if i == j:
                continue
            diff = abs(avg_ranks[i] - avg_ranks[j])
            q = diff / np.sqrt(k * (k + 1) / (6 * N))
            q_matrix[i, j] = q

    # aproximação normal: p ≈ 2 * (1 - Φ(q * sqrt(2)))
    p_matrix = 2 * (1 - norm.cdf(q_matrix * np.sqrt(2.0)))

    return avg_ranks, p_matrix

# ------------------------------------------------------------
# DISPONIBILIDADE DE MÉTODOS POR CLASSIFICADOR
# ------------------------------------------------------------

def _available_methods(classifier: str) -> dict:
    """
    Carrega automaticamente os DataFrames para um classificador:
    - classifier: 'RANDOM' ou 'XGBOOST'
    Espera encontrar:
      df_lime_metrics_<CLASS>
      df_shape_metrics_<CLASS>
      df_anchor_metrics_<CLASS>
      df_permutation_metrics_<CLASS>
      df_surrogate_tree_metrics_<CLASS>
    (e variações de SHAP)
    """
    env = globals()
    suf = classifier.upper()

    def _get(name):
        df = env.get(name, None)
        if isinstance(df, pd.DataFrame) and not df.empty:
            return _drop_index_cols(df)
        return None

    def pick(prefix):
        return _get(f"{prefix}_{suf}")

    mapping = {
        "LIME":      pick("df_lime_metrics"),
        "SHAPLEY":   pick("df_shape_metrics"),
        "ANCHOR":    pick("df_anchor_metrics"),
        "PERM IMP":  pick("df_permutation_metrics"),
        "SURR TREE": pick("df_surrogate_tree_metrics"),
        # "PROTODASH": pick("df_protodash_metrics"),
    }

    # tenta variações SHAP se o principal não existir
    if mapping.get("SHAPLEY") is None:
        alt = env.get(f"df_shap_metrics_{suf}", None) or env.get(f"df_shapley_metrics_{suf}", None)
        if isinstance(alt, pd.DataFrame) and not alt.empty:
            mapping["SHAPLEY"] = _drop_index_cols(alt)

    return {k: v for k, v in mapping.items() if v is not None}


def _metrics_to_process(method_dfs: dict) -> list[str]:
    all_cols = set()
    for df in method_dfs.values():
        all_cols.update(df.columns)
    ordered = [m for m in METRIC_WHITELIST if m in all_cols]
    return ordered or sorted(list(all_cols))

# ------------------------------------------------------------
# CÁLCULO NÃO-PARAMÉTRICO POR MÉTRICA (FRIEDMAN + NEMENYI)
# ------------------------------------------------------------

def compute_nonparametric(method_dfs: dict, metric: str):
    """
    Calcula:
      - stats_desc: mean, sd por método
      - Friedman χ², p
      - Kendall W
      - Nemenyi pares (diff de média + p_Nemenyi)
      - N (número de instâncias usadas)
    """
    values = {}

    for method_name, df in method_dfs.items():
        if metric not in df.columns:
            continue

        col = df[metric]

        # converte cada célula para a média numérica
        arr = col.apply(_extract_mean_from_ci).to_numpy()
        arr = arr[np.isfinite(arr)]

        if FIRST_N is not None:
            arr = arr[:FIRST_N]

        if len(arr) > 0:
            values[method_name] = arr

    if len(values) < 2:
        return None  # sem dados suficientes

    # corta todos para o mesmo N
    values = _normalize_arrays(values)
    methods = list(values.keys())
    K = len(methods)
    N = len(values[methods[0]])

    # ---------- médias / sd ----------
    stats_desc = {
        m: (float(values[m].mean()), float(values[m].std(ddof=1)))
        for m in methods
    }

    # ---------- Friedman ----------
    fried_chi2, fried_p = friedmanchisquare(*values.values())

    # ---------- Ranks ----------
    rank_matrix = np.vstack(
        [rankdata(values[m]) for m in methods]
    ).T  # N x K

    # Kendall W (global)
    avg_rank_per_row = rank_matrix.mean(axis=1)
    S = np.sum((avg_rank_per_row - (K + 1) / 2.0) ** 2)
    kendall_W = 12 * S / (N * K * (K * K - 1))

    # ---------- Nemenyi ----------
    avg_ranks, p_matrix = nemenyi_test(rank_matrix)

    # ---------- Diferenças de média (para “forest plot”) ----------
    mean_diff_rows = []
    for i, g1 in enumerate(methods):
        for j, g2 in enumerate(methods):
            if i <= j:
                continue
            m1 = stats_desc[g1][0]
            m2 = stats_desc[g2][0]
            diff = m1 - m2           # g1 - g2
            p_val = p_matrix[i, j]   # p-valor Nemenyi
            mean_diff_rows.append((g1, g2, diff, p_val))

    mean_diff_df = pd.DataFrame(
        mean_diff_rows,
        columns=["group1", "group2", "meandiff", "p_nemenyi"]
    ).sort_values("meandiff")

    return {
        "stats": stats_desc,
        "friedman": {"chi2": fried_chi2, "p": fried_p},
        "kendall_W": kendall_W,
        "methods": methods,
        "avg_ranks": avg_ranks,
        "nemenyi_pairs": mean_diff_df,
        "N": N,
    }

# ------------------------------------------------------------
# COLUNA 5: DUTO PNG + RANKING DOS MÉTODOS
# ------------------------------------------------------------

def _draw_duto_ranking(ax, stats_desc: dict, image_path: str = DUTO_IMG):
    """
    Coluna 5: duto 3D com PNG + rótulos dos métodos ordenados
    por média (maior -> menor).
    Usa stats_desc[m] = (mean, sd).
    """
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    if not stats_desc:
        ax.text(0.02, 0.5, "n/a", fontsize=8, va="center", ha="left")
        return

    medias = {m: v[0] for m, v in stats_desc.items() if np.isfinite(v[0])}
    if not medias:
        ax.text(0.02, 0.5, "n/a", fontsize=8, va="center", ha="left")
        return

    ordered = sorted(medias.items(), key=lambda kv: kv[1], reverse=True)
    methods = [m.upper() for m, _ in ordered]

    if not os.path.exists(image_path):
        ax.text(0.02, 0.5, f"Imagem não encontrada:\n{image_path}",
                fontsize=8, va="center", ha="left")
        return

    img = mpimg.imread(image_path)

    # posição do duto na área 0..1
    left, right = 0.04, 0.34
    bottom, top = 0.05, 0.95
    ax.imshow(img, extent=[left, right, bottom, top],
              aspect="auto", zorder=-1)

    # + e - ao lado
    sym_x = right + 0.05
    ax.text(sym_x, top - 0.01, "+", fontsize=10, fontweight="bold",
            ha="center", va="top", color="black")
    ax.text(sym_x, bottom + 0.01, "−", fontsize=10, fontweight="bold",
            ha="center", va="bottom", color="black")

    # labels ordenados verticalmente
    label_x = right + 0.14
    y_top = top - 0.08
    y_bot = bottom + 0.08
    ys = np.linspace(y_top, y_bot, len(methods))
    for y, m in zip(ys, methods):
        ax.text(label_x, y, m, fontsize=7, fontweight="bold",
                ha="left", va="center", color="black")

# ------------------------------------------------------------
# DESENHO 1 LINHA (5 QUADRANTES + 3 ESPAÇADORES)
# ------------------------------------------------------------

def draw_metric_row(fig, gs_row, metric_name: str, method_dfs: dict):
    res = compute_nonparametric(method_dfs, metric_name)
    if res is None:
        ax = fig.add_subplot(gs_row)
        ax.axis("off")
        ax.text(0.5, 0.5, f"Sem dados suficientes para {metric_name}",
                ha="center", va="center")
        return

    stats_desc   = res["stats"]
    fried        = res["friedman"]
    W            = res["kendall_W"]
    methods      = res["methods"]
    avg_ranks    = res["avg_ranks"]
    mean_diff_df = res["nemenyi_pairs"]
    N            = res["N"]

    # 8 colunas: [C1][SP12][C2][C3][SP34][C4][SP45][C5]
    inner = gs_row.subgridspec(
        nrows=1, ncols=8, wspace=0.06,
        width_ratios=[1.70, 0.99, 1.70, 1.55, 0.90, 1.50, 0.30, 1.25]
    )

    ax1 = fig.add_subplot(inner[0, 0])  # Barras
    ax2 = fig.add_subplot(inner[0, 2])  # “Tukey-style” Nemenyi
    ax3 = fig.add_subplot(inner[0, 3])  # Texto Friedman + Nemenyi
    ax4 = fig.add_subplot(inner[0, 5])  # Kendall W (tamanho de efeito)
    ax5 = fig.add_subplot(inner[0, 7])  # Duto + ranking

    # ---------- C1: Barras (média ± SD) ----------
    ax1.set_title(f"Métrica: {metric_name}", loc="left",
                  fontsize=9, fontweight="bold", pad=6)

    xs = np.arange(len(methods))
    means = [stats_desc[m][0] for m in methods]
    sds   = [stats_desc[m][1] for m in methods]
    colors = [METHOD_COLORS.get(m, "#444444") for m in methods]

    bars = ax1.bar(xs, means, yerr=sds, capsize=3,
                   color=colors, edgecolor="white",
                   linewidth=0.6, zorder=2)

    for x_i, b, m in zip(xs, bars, methods):
        ax1.text(x_i, b.get_height() * 0.02,
                 METHOD_LABELS.get(m, m),
                 rotation=90, ha="center", va="bottom",
                 fontsize=7, color="black",
                 clip_on=True, zorder=3)

    ax1.set_xticks([])
    ax1.set_ylabel(f"Média (todas as instâncias: {N})", fontsize=8)
    ax1.grid(True, axis="y", linestyle="--", alpha=0.32, zorder=1)
    ax1.set_xlim(-0.55, len(methods) - 0.45)
    ax1.margins(x=0.0)

    # ---------- C2: “Tukey-style” Nemenyi ----------
    ax2.axvline(0, color="gray", ls="--", lw=0.9)
    ax2.set_title("Nemenyi — pares (Δ média)", fontsize=8, pad=4)

    if mean_diff_df.empty:
        ax2.set_yticks([])
        ax2.set_xlabel("Δ média", fontsize=8)
    else:
        pairs = mean_diff_df.reset_index(drop=True)
        y = np.arange(len(pairs))

        diffs = pairs["meandiff"].to_numpy(dtype=float)
        pvals = pairs["p_nemenyi"].to_numpy(dtype=float)

        # intervalo visual simples: ±10% do max |diff|
        if np.any(np.isfinite(diffs)):
            span = max(1e-9, np.nanmax(np.abs(diffs)))
        else:
            span = 1.0
        half = 0.10 * span
        lower = diffs - half
        upper = diffs + half

        sig_mask = pvals < 0.05

        # segmentos
        ax2.hlines(y, lower, upper,
                   color="black", lw=1.2, zorder=1)
        ax2.hlines(y[sig_mask], lower[sig_mask], upper[sig_mask],
                   color="#2ca02c", lw=2.0, zorder=2)

        # pontos
        ax2.plot(diffs, y, "ko", ms=3.5, zorder=3)

        labels = (pairs["group1"] + " – " + pairs["group2"]).tolist()
        ax2.set_yticks(y)
        ax2.set_yticklabels(labels, fontsize=6)
        ax2.tick_params(axis="y", pad=1)

        ax2.set_xlabel("Diferença de média (g1 - g2) | verde = p<0.05 (Nemenyi)",
                       fontsize=8)

    # ---------- C3: Texto Friedman + tabela Nemenyi ----------
    ax3.axis("off")

    txt_lines = []
    txt_lines.append(
        f"Friedman: χ² = {fried['chi2']:.4f},  p = {fried['p']:.8f}"
    )
    txt_lines.append(f"Kendall W = {W:.4f}")
    txt_lines.append("")
    txt_lines.append("Nemenyi (pares):")

    if mean_diff_df.empty:
        txt_lines.append("  n/a")
    else:
        header = "group1   group2   meandiff   p_nem    sig"
        txt_lines.append(header)
        txt_lines.append("-" * len(header))
        for _, row in mean_diff_df.iterrows():
            g1 = str(row["group1"])
            g2 = str(row["group2"])
            md = float(row["meandiff"])
            pv = float(row["p_nemenyi"])
            sig = "True" if pv < 0.05 else "False"
            txt_lines.append(
                f"{g1:<8} {g2:<8} {md:>8.2f} {pv:>8.4f} {sig:>5}"
            )

    ax3.text(0.0, 1.0, "\n".join(txt_lines),
             va="top", ha="left",
             family="monospace", fontsize=6.7)

    # ---------- C4: Kendall W (tamanho de efeito global) ----------
    ax4.set_xlim(0.0, 1.0)
    ax4.set_ylim(0, 1)
    ax4.set_yticks([])
    ax4.set_xlabel("Tamanho de efeito (Kendall W)", fontsize=8)
    ax4.set_title("W — categorias", fontsize=8, pad=4)

    for x_thr, rotulo in [(0.1, "0.1"), (0.3, "0.3"), (0.5, "0.5")]:
        ax4.axvline(x_thr, color="#bdbdbd", lw=1.2)
        ax4.text(x_thr, 0.98, rotulo, rotation=90,
                 fontsize=7, ha="center", va="top", color="#6b6b6b")

    cat, x_cat = _cohen_bucket_W(W)
    if np.isfinite(W):
        ax4.plot(x_cat, 0.5, "o", color="#1f77b4", ms=5)
        ax4.text(x_cat, 0.5, f" W={W:.3f}\n({cat})",
                 fontsize=7.5, va="center", ha="left", color="#1f77b4")

    # ---------- C5: Duto + ranking ----------
    _draw_duto_ranking(ax5, stats_desc, image_path=DUTO_IMG)

# ------------------------------------------------------------
# GERAÇÃO DO PDF POR CLASSIFICADOR
# ------------------------------------------------------------

def build_dashboard_pdf(classifier: str):
    method_dfs = _available_methods(classifier)
    if not method_dfs:
        print(f"⚠️ Nenhum DF encontrado para {classifier}.")
        return

    metrics = _metrics_to_process(method_dfs)
    if not metrics:
        print(f"⚠️ Nenhuma métrica reconhecida encontrada para {classifier}.")
        return

    output_path = output_path_for(classifier)
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)

    with PdfPages(output_path) as pdf:
        for start in range(0, len(metrics), ROWS_PER_PAGE):
            subset = metrics[start:start + ROWS_PER_PAGE]

            fig = plt.figure(figsize=PAGE_SIZE)
            fig.subplots_adjust(left=0.075, right=0.975,
                                top=0.90, bottom=0.06)

            outer = fig.add_gridspec(
                nrows=len(subset), ncols=1, hspace=0.44
            )

            for r, metric in enumerate(subset):
                draw_metric_row(fig, outer[r, 0], metric, method_dfs)

            fig.suptitle(
                f"Comparação de Métricas — Friedman, Nemenyi, Kendall W e Ranking (semáforo)\nClassificador: {classifier}",
                x=0.008, y=0.965, ha="left",
                fontsize=10.5, fontweight="bold"
            )

            pdf.savefig(fig)
            plt.close(fig)

    print(f"✅ PDF salvo em: {output_path}")

# ------------------------------------------------------------
# EXECUTAR PARA RANDOM E XGBOOST
# ------------------------------------------------------------

build_dashboard_pdf("RANDOM")
build_dashboard_pdf("XGBOOST")


✅ PDF salvo em: /5_xAI/databases/FHS/notebooks/Methods/XAI_analisys/pdf_metricas/dashboard_metricas_RANDOM_friedman_nemenyi_duto.pdf
✅ PDF salvo em: /5_xAI/databases/FHS/notebooks/Methods/XAI_analisys/pdf_metricas/dashboard_metricas_XGBOOST_friedman_nemenyi_duto.pdf
